# Tarea 2 — Escalabilidad de KNN con datos médicos

**Análisis de comportamiento computacional y escalabilidad** de KNN (Cleveland Heart Disease) a partir de los resultados generados por `knn_heart_sklearn_scale.py` y orquestados por `knn_heart.sh`.

El barrido contenido en `results_knn_heart.csv` cubre las siguientes dimensiones (ver `knn_heart.sh`):

- `n_jobs`     ∈ {1, 2, 4, 8, 16, 32}        — hilos (backend `threading`, algoritmo `brute`).
- `mult_train` ∈ {1, 4, 16, 64, 256, 1024}   — factor de expansión del set de entrenamiento.
- `mult_test`  ∈ {1, 16, 128}                — factor de expansión del set de prueba.
- `feat_mult`  ∈ {1, 2, 4, 8}                — factor de expansión de atributos (modo `mix`).
- `k=5` fijo.

**Métrica principal:** `pred_time_s_avg` (en KNN brute, `fit` es lazy — el trabajo real ocurre en `predict`).

Complejidad teórica de KNN brute:

$$T(n_{\text{train}}, n_{\text{test}}, d, k, p) = \Theta\!\left(\frac{n_{\text{test}} \cdot n_{\text{train}} \cdot d}{p}\right)$$

donde el término $n_{\text{test}} \cdot n_{\text{train}} \cdot d$ corresponde al cómputo de distancias (dominante) y se busca paralelizar sobre $p$ hilos.

## 1. Setup y carga de datos

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

CSV_PATH = Path("results_knn_heart.csv")
df = pd.read_csv(CSV_PATH)
df["total_time"] = df["fit_time_s_avg"] + df["pred_time_s_avg"]
for col in ["flops_total", "flops_distance", "flops_select"]:
    df[col] = df[col].astype(float)

# Ejes únicos del barrido
N_JOBS  = sorted(df["n_jobs"].unique())
N_TRAIN = sorted(df["n_train"].unique())
N_TEST  = sorted(df["n_test"].unique())
N_FEAT  = sorted(df["n_features"].unique())

# Configuraciones "grande" y "mediano" usadas en §3 y §5
N_TRAIN_BIG, N_FEAT_BIG, N_TEST_BIG = N_TRAIN[-1], N_FEAT[-1], N_TEST[-1]
N_TRAIN_MED, N_FEAT_MED, N_TEST_MED = N_TRAIN[len(N_TRAIN)//2], N_FEAT[len(N_FEAT)//2], N_TEST[len(N_TEST)//2]

print("Filas:", len(df))
print("n_jobs:    ", N_JOBS)
print("n_train:   ", N_TRAIN)
print("n_test:    ", N_TEST)
print("n_features:", N_FEAT)
print("backend / algorithm:", df["backend"].unique(), df["algorithm"].unique())
print(f"\nCaso grande:  n_train={N_TRAIN_BIG}, n_features={N_FEAT_BIG}, n_test={N_TEST_BIG}")
print(f"Caso mediano: n_train={N_TRAIN_MED}, n_features={N_FEAT_MED}, n_test={N_TEST_MED}")
df.head()

## 2. Costo computacional al variar el tamaño del problema

Para aislar el efecto del tamaño del problema, fijamos `n_jobs=1` (ejecución serial) y, salvo el eje variado, las dimensiones se fijan en su valor máximo del barrido.

_TODO: completar análisis._

In [ ]:
# Gráfica 2.1: tiempo vs n_train (una curva por n_features), n_jobs=1, n_test=N_TEST_BIG
serial = df[df["n_jobs"] == 1]
sub_g = serial[serial["n_test"] == N_TEST_BIG]

fig, ax = plt.subplots()
for d_val, sub in sub_g.groupby("n_features"):
    sub = sub.sort_values("n_train")
    ax.loglog(sub["n_train"], sub["pred_time_s_avg"], "o-", label=f"d={d_val}")

ref_x = np.array([sub_g["n_train"].min(), sub_g["n_train"].max()])
anchor = sub_g[(sub_g["n_features"] == N_FEAT_BIG) & (sub_g["n_train"] == ref_x[0])]["pred_time_s_avg"].iloc[0]
ax.loglog(ref_x, anchor * ref_x / ref_x[0], "k--", alpha=0.5, label="O(n) referencia")

ax.set_xlabel("n_train")
ax.set_ylabel("pred_time_s_avg [s]")
ax.set_title(f"Tiempo de predicción vs número de muestras (n_jobs=1, n_test={N_TEST_BIG})")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Gráfica 2.2: tiempo vs n_features (una curva por n_train), n_jobs=1, n_test=N_TEST_BIG
fig, ax = plt.subplots()
for n_val, sub in sub_g.groupby("n_train"):
    sub = sub.sort_values("n_features")
    ax.loglog(sub["n_features"], sub["pred_time_s_avg"], "o-", label=f"n_train={n_val}")

ref_x = np.array([sub_g["n_features"].min(), sub_g["n_features"].max()])
anchor = sub_g[(sub_g["n_train"] == N_TRAIN_BIG) & (sub_g["n_features"] == ref_x[0])]["pred_time_s_avg"].iloc[0]
ax.loglog(ref_x, anchor * ref_x / ref_x[0], "k--", alpha=0.5, label="O(d) referencia")

ax.set_xlabel("n_features (d)")
ax.set_ylabel("pred_time_s_avg [s]")
ax.set_title(f"Tiempo de predicción vs número de atributos (n_jobs=1, n_test={N_TEST_BIG})")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Gráfica 2.3: tiempo vs n_test (una curva por n_train), n_jobs=1, n_features=N_FEAT_BIG
sub_h = serial[serial["n_features"] == N_FEAT_BIG]

fig, ax = plt.subplots()
for n_val, sub in sub_h.groupby("n_train"):
    sub = sub.sort_values("n_test")
    ax.loglog(sub["n_test"], sub["pred_time_s_avg"], "o-", label=f"n_train={n_val}")

ref_x = np.array([sub_h["n_test"].min(), sub_h["n_test"].max()])
anchor = sub_h[(sub_h["n_train"] == N_TRAIN_BIG) & (sub_h["n_test"] == ref_x[0])]["pred_time_s_avg"].iloc[0]
ax.loglog(ref_x, anchor * ref_x / ref_x[0], "k--", alpha=0.5, label="O(n_test) referencia")

ax.set_xlabel("n_test")
ax.set_ylabel("pred_time_s_avg [s]")
ax.set_title(f"Tiempo de predicción vs número de queries de prueba (n_jobs=1, d={N_FEAT_BIG})")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Gráfica 2.4: tiempo vs FLOPs teóricos (todos los puntos serial)
fig, ax = plt.subplots()
ax.loglog(serial["flops_total"], serial["pred_time_s_avg"], "o", alpha=0.7, label="Medidos")

log_x = np.log(serial["flops_total"].values)
log_y = np.log(serial["pred_time_s_avg"].values)
slope, intercept = np.polyfit(log_x, log_y, 1)
x_fit = np.array([serial["flops_total"].min(), serial["flops_total"].max()])
ax.loglog(x_fit, np.exp(intercept) * x_fit ** slope, "r--",
          label=f"fit log-log, pendiente={slope:.3f}")

ax.set_xlabel("FLOPs totales (teóricos)")
ax.set_ylabel("pred_time_s_avg [s]")
ax.set_title("Tiempo medido vs FLOPs teóricos (n_jobs=1)")
ax.legend()
plt.tight_layout()
plt.show()

print(f"Pendiente del fit: {slope:.4f} (esperado ≈ 1 si T ∝ FLOPs)")

_TODO: análisis de la sección 2 (escalado con muestras, atributos y queries)._

## 3. Escalabilidad fuerte (strong scaling)

**Definición.** Problema fijo, variamos `n_jobs`. Métricas:

- $T(p) =$ `pred_time_s_avg` con `n_jobs = p`.
- $\text{Speedup}(p) = T(1) / T(p)$.
- $\text{Eficiencia}(p) = \text{Speedup}(p) / p$.

Configuraciones analizadas:

1. **Caso grande**: `n_train=N_TRAIN_BIG`, `n_features=N_FEAT_BIG`, `n_test=N_TEST_BIG` — el más costoso del barrido.
2. **Caso mediano**: `n_train=N_TRAIN_MED`, `n_features=N_FEAT_MED`, `n_test=N_TEST_MED`.

_TODO: completar análisis._

In [ ]:
def strong_scaling(df, n_train, n_features, n_test):
    sub = df[(df["n_train"] == n_train) &
             (df["n_features"] == n_features) &
             (df["n_test"] == n_test)].copy()
    sub = sub.sort_values("n_jobs").reset_index(drop=True)
    T1 = sub.loc[sub["n_jobs"] == 1, "pred_time_s_avg"].iloc[0]
    sub["speedup"] = T1 / sub["pred_time_s_avg"]
    sub["efficiency"] = sub["speedup"] / sub["n_jobs"]
    return sub[["n_jobs", "pred_time_s_avg", "speedup", "efficiency"]]

big = strong_scaling(df, N_TRAIN_BIG, N_FEAT_BIG, N_TEST_BIG)
med = strong_scaling(df, N_TRAIN_MED, N_FEAT_MED, N_TEST_MED)

LABEL_BIG = f"grande ({N_TRAIN_BIG}×{N_FEAT_BIG}, n_test={N_TEST_BIG})"
LABEL_MED = f"mediano ({N_TRAIN_MED}×{N_FEAT_MED}, n_test={N_TEST_MED})"

print(f"Caso {LABEL_BIG}:")
print(big.to_string(index=False))
print(f"\nCaso {LABEL_MED}:")
print(med.to_string(index=False))

In [ ]:
# Gráfica 3.1: tiempo paralelo vs n_jobs (log-log) + curva ideal T(1)/p
fig, ax = plt.subplots()
for label, sub in [(LABEL_BIG, big), (LABEL_MED, med)]:
    ax.loglog(sub["n_jobs"], sub["pred_time_s_avg"], "o-", label=f"Medido — {label}")
    T1 = sub["pred_time_s_avg"].iloc[0]
    ax.loglog(sub["n_jobs"], T1 / sub["n_jobs"], "--", alpha=0.5,
              label=f"Ideal T(1)/p — {label}")

ax.set_xlabel("n_jobs (p)")
ax.set_ylabel("pred_time_s_avg [s]")
ax.set_title("Strong scaling: tiempo de ejecución paralelo vs n_jobs")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Gráfica 3.2: speedup vs n_jobs
fig, ax = plt.subplots()
for label, sub in [(LABEL_BIG, big), (LABEL_MED, med)]:
    ax.plot(sub["n_jobs"], sub["speedup"], "o-", label=label)

ax.plot(N_JOBS, N_JOBS, "k--", alpha=0.5, label="Ideal (Speedup = p)")

ax.set_xlabel("n_jobs (p)")
ax.set_ylabel("Speedup")
ax.set_title("Strong scaling: Speedup vs n_jobs")
ax.set_xscale("log", base=2)
ax.set_yscale("log", base=2)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Gráfica 3.3: eficiencia vs n_jobs
fig, ax = plt.subplots()
for label, sub in [(LABEL_BIG, big), (LABEL_MED, med)]:
    ax.plot(sub["n_jobs"], sub["efficiency"], "o-", label=label)

ax.axhline(1.0, color="k", linestyle="--", alpha=0.5, label="Ideal (Eficiencia = 1)")
ax.set_xlabel("n_jobs (p)")
ax.set_ylabel("Eficiencia")
ax.set_title("Strong scaling: Eficiencia vs n_jobs")
ax.set_xscale("log", base=2)
ax.set_ylim(0, 1.2)
ax.legend()
plt.tight_layout()
plt.show()

_TODO: análisis de la sección 3 (strong scaling)._

## 4. Escalabilidad débil (weak scaling)

**Definición.** El trabajo por hilo se mantiene constante: emparejamos `n_jobs` con `n_train` por índice del barrido. Fijamos `n_test = N_TEST_BIG`.

Métricas (con $T_w(p)$ el tiempo de predicción de la pareja):

- $\text{Speedup débil}(p) = p \cdot T_w(1) / T_w(p)$ (ideal = $p$).
- $\text{Eficiencia débil}(p) = T_w(1) / T_w(p)$ (ideal = 1).

_TODO: completar análisis._

In [ ]:
pairs = list(zip(N_JOBS, N_TRAIN))
print("Emparejamientos (p, n_train):", pairs)

def weak_scaling(df, n_features, n_test):
    rows = []
    for p, n in pairs:
        r = df[(df["n_jobs"] == p) &
               (df["n_train"] == n) &
               (df["n_features"] == n_features) &
               (df["n_test"] == n_test)]
        rows.append({"n_jobs": p, "n_train": n, "pred_time": r["pred_time_s_avg"].iloc[0]})
    out = pd.DataFrame(rows)
    T1 = out["pred_time"].iloc[0]
    out["weak_speedup"]    = T1 * out["n_jobs"] / out["pred_time"]
    out["weak_efficiency"] = T1 / out["pred_time"]
    return out

weak_curves = {d: weak_scaling(df, d, N_TEST_BIG) for d in N_FEAT}
for d, w in weak_curves.items():
    print(f"\nWeak scaling, d={d} (n_test={N_TEST_BIG}):")
    print(w.to_string(index=False))

In [ ]:
# Gráfica 4.1: tiempo de ejecución paralelo vs n_jobs (carga proporcional, ideal = plano)
fig, ax = plt.subplots()
for d, w in weak_curves.items():
    ax.plot(w["n_jobs"], w["pred_time"], "o-", label=f"d={d}")

ax.set_xscale("log", base=2)
ax.set_xlabel("n_jobs (p) — n_train escalado proporcionalmente")
ax.set_ylabel("pred_time_s_avg [s]")
ax.set_title(f"Weak scaling: tiempo de ejecución paralelo (n_test={N_TEST_BIG})")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Gráfica 4.2: speedup débil vs n_jobs
fig, ax = plt.subplots()
for d, w in weak_curves.items():
    ax.plot(w["n_jobs"], w["weak_speedup"], "o-", label=f"d={d}")

ax.plot(N_JOBS, N_JOBS, "k--", alpha=0.5, label="Ideal (Speedup = p)")

ax.set_xscale("log", base=2)
ax.set_yscale("log", base=2)
ax.set_xlabel("n_jobs (p)")
ax.set_ylabel("Speedup débil = p · T(1) / T(p)")
ax.set_title(f"Weak scaling: Speedup vs n_jobs (n_test={N_TEST_BIG})")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Gráfica 4.3: eficiencia débil vs n_jobs
fig, ax = plt.subplots()
for d, w in weak_curves.items():
    ax.plot(w["n_jobs"], w["weak_efficiency"], "o-", label=f"d={d}")

ax.axhline(1.0, color="k", linestyle="--", alpha=0.5, label="Ideal (= 1)")
ax.set_xscale("log", base=2)
ax.set_xlabel("n_jobs (p)")
ax.set_ylabel("Eficiencia débil = T(1) / T(p)")
ax.set_title(f"Weak scaling: Eficiencia vs n_jobs (n_test={N_TEST_BIG})")
ax.legend()
plt.tight_layout()
plt.show()

_TODO: análisis de la sección 4 (weak scaling)._

## 5. Overhead

Definimos:

- **Overhead absoluto:** $O(p) = T(p) - T(1)/p$ — exceso sobre la curva ideal.
- **Fracción de overhead:** $O(p) / T(p)$ — qué proporción del tiempo medido es sobrecarga.

_TODO: completar análisis._

In [ ]:
def overhead_table(strong_df):
    out = strong_df.copy()
    T1 = out["pred_time_s_avg"].iloc[0]
    out["ideal"] = T1 / out["n_jobs"]
    out["overhead_abs"]  = out["pred_time_s_avg"] - out["ideal"]
    out["overhead_frac"] = out["overhead_abs"] / out["pred_time_s_avg"]
    return out

oh_big = overhead_table(big)
oh_med = overhead_table(med)
print(f"Caso {LABEL_BIG}:")
print(oh_big.to_string(index=False))
print(f"\nCaso {LABEL_MED}:")
print(oh_med.to_string(index=False))

In [ ]:
# Gráfica 5.1: overhead absoluto
fig, ax = plt.subplots()
ax.plot(oh_big["n_jobs"], oh_big["overhead_abs"], "o-", label=LABEL_BIG)
ax.plot(oh_med["n_jobs"], oh_med["overhead_abs"], "o-", label=LABEL_MED)
ax.set_xscale("log", base=2)
ax.set_xlabel("n_jobs (p)")
ax.set_ylabel("Overhead absoluto = T(p) - T(1)/p  [s]")
ax.set_title("Overhead absoluto vs n_jobs")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Gráfica 5.2: fracción de overhead
fig, ax = plt.subplots()
ax.plot(oh_big["n_jobs"], oh_big["overhead_frac"], "o-", label=LABEL_BIG)
ax.plot(oh_med["n_jobs"], oh_med["overhead_frac"], "o-", label=LABEL_MED)
ax.set_xscale("log", base=2)
ax.set_xlabel("n_jobs (p)")
ax.set_ylabel("Fracción de overhead")
ax.set_title("Fracción del tiempo medido que es overhead")
ax.set_ylim(-0.05, 1.05)
ax.legend()
plt.tight_layout()
plt.show()

_TODO: análisis de la sección 5 (overhead)._

## 6. Comparación con la complejidad teórica

Confirmamos que el tiempo medido sigue la forma teórica $T \propto n_{\text{test}} \cdot n_{\text{train}} \cdot d$ (proporcional a `flops_total`).

_TODO: completar análisis._

In [ ]:
# Gráfica 6.1: tiempo vs flops_total con fit lineal (n_jobs=1, todos los tamaños)
fig, ax = plt.subplots()
ax.loglog(serial["flops_total"], serial["pred_time_s_avg"], "o", alpha=0.7, label="Medidos")

log_x = np.log(serial["flops_total"].values)
log_y = np.log(serial["pred_time_s_avg"].values)
slope, intercept = np.polyfit(log_x, log_y, 1)
x_fit = np.array([serial["flops_total"].min(), serial["flops_total"].max()])
ax.loglog(x_fit, np.exp(intercept) * x_fit ** slope, "r--",
          label=f"fit: T ∝ FLOPs^{slope:.3f}")

ax.set_xlabel("FLOPs totales (teóricos)")
ax.set_ylabel("pred_time_s_avg [s]")
ax.set_title("Tiempo medido vs FLOPs teóricos")
ax.legend()
plt.tight_layout()
plt.show()

throughput_serial = serial["flops_total"].sum() / serial["pred_time_s_avg"].sum()
print(f"Pendiente del fit: {slope:.4f} (esperado ≈ 1)")
print(f"Throughput promedio (serial): {throughput_serial:.2e} FLOP/s")

In [ ]:
# Gráfica 6.2: throughput efectivo vs n_jobs (caso grande)
big_full = df[(df["n_train"] == N_TRAIN_BIG) &
              (df["n_features"] == N_FEAT_BIG) &
              (df["n_test"] == N_TEST_BIG)].sort_values("n_jobs")
throughput = big_full["flops_total"] / big_full["pred_time_s_avg"]

fig, ax = plt.subplots()
ax.plot(big_full["n_jobs"], throughput, "o-")
ax.set_xscale("log", base=2)
ax.set_xlabel("n_jobs (p)")
ax.set_ylabel("Throughput efectivo [FLOP/s]")
ax.set_title(f"Throughput sostenido vs n_jobs ({LABEL_BIG})")
plt.tight_layout()
plt.show()

print("FLOPs/s por valor de n_jobs:")
for p, t in zip(big_full["n_jobs"], throughput):
    print(f"  p={p:>2}: {t:.2e} FLOP/s")

_TODO: análisis de la sección 6 (complejidad teórica)._

## 7. Conclusiones

_TODO: completar conclusiones._
